# $\gamma = I/H$ across a ladder of non-linearity

The third RPC metric. $\rho$ and $\lambda$ need no normalisation; $\gamma$ divides a mutual
information by an entropy, so it stands or falls on the denominator.

$$\gamma_o = \frac{I(s;g)}{H(g)},\qquad
\gamma_m = \frac1{|n|}\sum_n \frac{I(s_{-n};f_n)}{H(f_n)},\qquad
\mathrm{RPC}_\gamma = \frac{\gamma_o}{\gamma_m}$$

Each side normalises by the entropy of **its own** prediction target, so each $\gamma$ reads
as the fraction of that target's uncertainty the predictor accounts for. $\gamma$ is formed
per member and then averaged, as $\lambda$ is.

**The rule.** The numerator and the denominator must be the same kind of quantity from the
same estimator family over the same variables. One `level` string in
`R.GAMMA_LADDER` picks the MI approach, the entropy approach and the transform each sees,
so the two cannot drift apart.

| level | $I$ | $H$ | free parameter | denominator kind |
|---|---|---|---|---|
| `gaussian` | $-\tfrac12\log_2(1-r^2)$ | $\tfrac12\log_2 2\pi e$ | — | differential |
| `discrete` | `discrete` | `discrete` | `nbins` | Shannon |
| `miller_madow` | `miller_madow` | `miller_madow` | `nbins` | Shannon |
| `ordinal` | `ordinal` | `ordinal` | `embedding_dim` | Shannon |
| `kernel` | `kernel` | `kernel` | `bandwidth` | differential |
| `knn` | `ksg` | `metric` | `k` | differential |

A Shannon denominator is $\ge 0$ and bounded by $\log_2$ of the number of states, so
$\gamma\in[0,1]$. A differential denominator is unbounded below and shifts by
$\log_2(\text{scale})$ under a change of units, so those rungs standardise first, fixing
$H$ at $\tfrac12\log_2 2\pi e - J$ for negentropy $J\ge0$: a shape measure with a ceiling
of 2.047 bits.

Both treatments are computed throughout, since the preprocessing removes a linear trend and
no seasonal cycle.

**Handles**

| name | shape | meaning |
|---|---|---|
| `grid[level][t][field]` | `(64, 128)` | `field` from `R.GAMMA_FIELDS` |
| `GAM_O[level][t]`, `GAM_M[level][t]` | `(64, 128)` | $\gamma_o$, $\gamma_m$ |
| `RPC[level][t]` | `(64, 128)` | $\gamma_o/\gamma_m$ |
| `rho_ratio_var` | `(64, 128)` | published variance-based $\rho_o/\rho_m$ |

`t` is one of `TREATMENTS`; `panel(ax, field, cmap, norm, title)` draws a map.

In [ ]:
import os
import sys
import warnings
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc
from matplotlib.colors import Normalize
import cartopy.crs as ccrs

sys.path.insert(0, "../scripts")
import infomeasure as im
import infomeasure_rpc as R

rc('font', **{'family': 'sans-serif', 'serif': ['Helvetica']})
rc('text', usetex=False)

DATA = "../data" if os.path.isdir("../data/ensembles") else \
    "/Users/jasperchen/Academics/Research/SNP/information-snp/data"
f, s, obs = R.load_data(data_dir=DATA)
lats, lons = f["lat"].to_numpy(), f["lon"].to_numpy()

# Repair the two defective edge longitude columns. In the raw ensemble file members
# 10-19 have no data at all at lon index 0 (0.00E) and members 20-39 none at index 127
# (357.19E) -- whole model groups missing one edge column, from regridding onto the
# common grid without periodic longitude handling. Zero-filled they read as a stripe
# down the middle of every rolled map. Fill each from its nearest longitude neighbour;
# timesteps missing for other reasons (calendar 1971 for members 10-19) stay NaN.
_psl = f.psl.to_numpy()
for _bad, _good in ((0, 1), (127, 126)):
    _col = _psl[:, :, :, _bad]
    _psl[:, :, :, _bad] = np.where(np.isnan(_col), _psl[:, :, :, _good], _col)
f["psl"] = (f.psl.dims, _psl)
s = f.mean("n")

TREATMENTS = list(R.GAMMA_TREATMENTS)
LEVELS = list(R.GAMMA_LADDER)
PROBE = (32, 21)

# The variance-based RPC never touches an information estimator, so it is the check that
# the preprocessing is right: the published figure is 72.1% of the globe above 1.
rho_o_var, rho_m_var, _ratio = R.var_RPC(f, s, obs)
rho_ratio_var = _ratio.to_numpy()
assert abs(R.area_weighted_fraction(rho_ratio_var, lats) - 0.721) < 0.005

PROJ = ccrs.PlateCarree()
LON180 = ((lons + 180) % 360) - 180        # grid is 0..357E; roll to -180..180
ORDER = np.argsort(LON180)
EXTENT = [LON180[ORDER].min(), LON180[ORDER].max(), lats.min(), lats.max()]


def panel(ax, field, cmap, norm, title):
    ax.set_global()
    h = ax.imshow(field[:, ORDER], origin="lower", extent=EXTENT, transform=PROJ,
                  cmap=cmap, norm=norm)
    ax.coastlines(linewidth=0.4, color="0.6")
    ax.set_title(title, fontsize=12)
    return h

## Why the transforms are what they are

Three ways the denominator collapses, each checked rather than asserted in prose.

1. **Equiprobable rank bins.** `R.rank_bins` puts exactly $T/\text{nbins}$ samples in every
   bin by construction, so $H \equiv \log_2\text{nbins}$ at every grid point and $\gamma$ is
   $I$ rescaled by a constant. `R.width_bins` bins on equal width instead, which lets $H$
   respond to the shape of the distribution.
2. **The copula transform.** Rank-transforming to uniform marginals sends the true
   differential entropy to $h(\mathcal{U}[0,1]) = 0$, so a kNN estimator on ranks returns
   only its own bias. MI *is* invariant to monotone marginal transforms, so the copula step
   stays in the numerator where it is worth 4$\times$; the denominator sees standardised
   values instead.
3. **The analytic rung.** With both sides standardised, $H = \tfrac12\log_2 2\pi e$ exactly
   and $\gamma_{\text{gauss}}$ is a monotone function of $|r|$ alone. It is the only rung
   with a closed form to check the numerator against.

And one thing that cannot be done: $H$ is a function of the sample distribution of its
argument, so permuting time leaves it **exactly** unchanged for every rung but `ordinal`.
There is no entropy null. Denominator bias has to be addressed by choosing a bias-corrected
estimator, which is what the `miller_madow` rung is for.

In [ ]:
probe_obs = np.nan_to_num(obs.psl.to_numpy()[:, PROBE[0], PROBE[1]], nan=0.0)
probe_signal = np.nan_to_num(s.psl.to_numpy()[:, PROBE[0], PROBE[1]], nan=0.0)

for nbins in (3, 5, 10):
    assert np.isclose(im.entropy(R.rank_bins(probe_obs, nbins), approach="discrete",
                                 base=2), np.log2(nbins))

r_probe = float(np.corrcoef(probe_signal, probe_obs)[0, 1])
assert np.isclose(R.mi_bits_matched(probe_signal, probe_obs, "gaussian"),
                  -0.5 * np.log2(1 - r_probe ** 2))
assert np.isclose(R.entropy_bits(probe_obs, "gaussian"), R.H_GAUSS)

print("%-8s %14s %14s" % ("nbins", "H rank bins", "H width bins"))
for nbins in (3, 5, 10):
    print("%-8d %14.6f %14.6f"
          % (nbins, im.entropy(R.rank_bins(probe_obs, nbins), approach="discrete", base=2),
             im.entropy(R.width_bins(probe_obs, nbins), approach="discrete", base=2)))

print("\nkNN H of the copula transform: %.4f bits, against a true h(U[0,1]) of 0"
      % im.entropy(R.copula(probe_obs), approach="metric", k=5, base=2))

_rng = np.random.default_rng(0)
_shuffled = _rng.permutation(probe_obs)
print("\n%-14s %10s %10s %12s" % ("level", "H", "H permuted", "invariant"))
for level in LEVELS:
    a, b = R.entropy_bits(probe_obs, level), R.entropy_bits(_shuffled, level)
    print("%-14s %10.4f %10.4f %12s" % (level, a, b, np.isclose(a, b)))

## The ladder at one grid point

Lat 32 / lon 21, both treatments, each rung at its default free parameter. `gamma_point`
returns both sides plus a permutation null on each numerator; `excess` columns are
$\gamma$ rebuilt from $I - I_{\text{null}}$, which is the only version comparable across
rungs, since the bias floors differ several-fold.

In [ ]:
members_probe = f.psl.to_numpy()[:, :, PROBE[0], PROBE[1]]
obs_probe = np.nan_to_num(obs.psl.to_numpy()[:, PROBE[0], PROBE[1]], nan=0.0)
POST = {"as-is": None, "deseasonalised": R.deseason}

for treatment in TREATMENTS:
    print("-- %s --" % treatment)
    print("%-14s %6s %7s %7s %7s %7s %7s %7s %8s %9s"
          % ("level", "param", "I_o", "H_o", "gam_o", "I_m", "H_m", "gam_m",
             "RPC_gam", "RPC excl"))
    for level in LEVELS:
        res = R.gamma_point(members_probe, obs_probe, level, n_perm=40, seed=1,
                            post=POST[treatment])
        print("%-14s %6s %7.4f %7.4f %7.4f %7.4f %7.4f %7.4f %8.4f %9.4f"
              % (level, R.GAMMA_LADDER[level]["values"][
                     R.GAMMA_LADDER[level]["default_idx"]],
                 res["I_o"], res["H_o"], res["gamma_o"],
                 res["I_m"], res["H_m"], res["gamma_m"],
                 res["gamma_o"] / res["gamma_m"],
                 res["gamma_o_excess"] / res["gamma_m_excess"]))
    print()

## The grid

Every rung, both treatments, 20 permutations per numerator. A cold run is about six minutes
over eight processes, so it is cached.

Two ratios are reported. The plain one divides $\gamma_o$ by $\gamma_m$ as estimated. The
excess one rebuilds both from $I-I_{\text{null}}$ first, which is what makes rungs with
different bias floors comparable -- but a ratio of two near-zero excesses is arbitrarily
large and carries a sign, and the KSG numerator is signed by construction. So the excess
ratio is kept only where **both** sides clear their own null; elsewhere it is NaN, and the
fraction of the globe that survives is reported next to it.

In [ ]:
CACHE = "../data/gamma_info_cache.npz"

if os.path.exists(CACHE):
    grid = R.load_gamma_cache(CACHE)
else:
    grid = R.gamma_grid(f, obs, n_perm=20, processes=8)
    R.save_gamma_cache(CACHE, grid)

GAM_O = {lev: {t: grid[lev][t]["gamma_o"] for t in TREATMENTS} for lev in LEVELS}
GAM_M = {lev: {t: grid[lev][t]["gamma_m"] for t in TREATMENTS} for lev in LEVELS}
RPC = {lev: {t: GAM_O[lev][t] / GAM_M[lev][t] for t in TREATMENTS} for lev in LEVELS}


def cleared_ratio(numer, denom):
    """numer/denom, NaN wherever either side fails to clear its own null."""
    ok = (numer > 0) & (denom > 0)
    return np.where(ok, numer / np.where(ok, denom, 1.0), np.nan)


RPC_EXCESS = {lev: {t: cleared_ratio(grid[lev][t]["gamma_o_excess"],
                                     grid[lev][t]["gamma_m_excess"])
                    for t in TREATMENTS} for lev in LEVELS}

# Medians, not means: these are ratios, and a ratio with a small denominator has a
# tail heavy enough that the mean tracks the worst few cells rather than the field.
print("%-14s %-15s %8s %8s %8s %8s %9s %10s %8s"
      % ("level", "treatment", "gam_o", "gam_m", "RPC med", "% > 1",
         "excl med", "% > 1 excl", "% kept"))
for level in LEVELS:
    for treatment in TREATMENTS:
        excess = RPC_EXCESS[level][treatment]
        print("%-14s %-15s %8.4f %8.4f %8.4f %7.1f%% %9.4f %9.1f%% %7.1f%%"
              % (level, treatment, np.nanmean(GAM_O[level][treatment]),
                 np.nanmean(GAM_M[level][treatment]),
                 np.nanmedian(RPC[level][treatment]),
                 100 * R.area_weighted_fraction(RPC[level][treatment], lats),
                 np.nanmedian(excess),
                 100 * R.area_weighted_fraction(excess, lats),
                 100 * np.mean(np.isfinite(excess))))
print("\n%-14s %-15s %8s %8s %8.4f %7.1f%%"
      % ("rho published", "as-is", "", "", np.nanmedian(rho_ratio_var),
         100 * R.area_weighted_fraction(rho_ratio_var, lats)))

## $\mathrm{RPC}_\gamma$ up the ladder

Rows are levels, columns treatments, on the same `Normalize(0, 2)` scale the $\lambda$ and
$\rho$ ratios use.

In [ ]:
fig, axs = plt.subplots(len(LEVELS), 2, figsize=(9, 2.2 * len(LEVELS)),
                        subplot_kw={"projection": PROJ})
nrm_ratio = Normalize(0.0, 2.0)
for i, level in enumerate(LEVELS):
    for j, treatment in enumerate(TREATMENTS):
        h = panel(axs[i, j], RPC[level][treatment], "RdBu_r", nrm_ratio,
                  "$\\mathrm{RPC}_\\gamma$, %s, %s" % (level, treatment))
        fig.colorbar(h, ax=axs[i, j], fraction=0.02)
plt.tight_layout()
plt.show()

## One rung in full

$\gamma_o$, $\gamma_m$ and their ratio for `LEVEL`, alongside the published $\rho$ ratio.

In [ ]:
LEVEL = "knn"

fig, axs = plt.subplots(3, 2, figsize=(9, 7), subplot_kw={"projection": PROJ})
for j, treatment in enumerate(TREATMENTS):
    a, b = GAM_O[LEVEL][treatment], GAM_M[LEVEL][treatment]
    nrm = Normalize(min(0.0, np.nanmin(a), np.nanmin(b)), max(np.nanmax(a), np.nanmax(b)))
    h0 = panel(axs[0, j], a, "viridis", nrm, "$\\gamma_o$, %s" % treatment)
    h1 = panel(axs[1, j], b, "viridis", nrm, "$\\gamma_m$, %s" % treatment)
    h2 = panel(axs[2, j], a - b, "RdBu_r", Normalize(-0.3, 0.3),
               "$\\gamma_o - \\gamma_m$, %s" % treatment)
    fig.colorbar(h0, ax=axs[0, j], shrink=0.85, label="")
    fig.colorbar(h1, ax=axs[1, j], shrink=0.85, label="")
    fig.colorbar(h2, ax=axs[2, j], shrink=0.85, label="")
plt.tight_layout()
plt.show()

## $\mathrm{RPC}_\gamma$ against $\mathrm{RPC}_\rho$

The published variance-based ratio is the reference on both panels: a distribution of the
ratio up the ladder, and the linear rung against $\rho$ directly. The linear rung's
denominator is a constant, so its $\gamma$ is a monotone function of $|r|$ and this is the
one comparison with no estimator freedom in it.

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")

BINS = np.linspace(-1, 3, 60)
axs[0].hist(rho_ratio_var.ravel(), bins=BINS, density=True, histtype="stepfilled",
            alpha=0.4, color="k", label="$\\rho$, published")
for level in LEVELS:
    axs[0].hist(RPC[level]["as-is"].ravel(), bins=BINS, density=True, histtype="step",
                lw=1.2, label=level)
axs[0].axvline(1.0, color="k", ls="--", lw=0.8)
axs[0].set_xlabel("RPC, as-is")
axs[0].set_ylabel("density")
axs[0].legend(fontsize=8)

for level, colour in zip(("gaussian", "knn"), ("tab:blue", "tab:orange")):
    axs[1].scatter(rho_ratio_var.ravel(), RPC[level]["as-is"].ravel(), s=1, alpha=0.15,
                   color=colour, label=level)
axs[1].plot([0, 2], [0, 2], "k--", lw=0.8)
axs[1].set_xlabel("$\\rho_o/\\rho_m$")
axs[1].set_ylabel("$\\gamma_o/\\gamma_m$")
axs[1].set_xlim(0, 2)
axs[1].set_ylim(0, 3)
leg = axs[1].legend(fontsize=8, markerscale=6)
for lh in leg.legend_handles:
    lh.set_alpha(1)
plt.show()

print("%-14s %12s %12s" % ("level", "corr with rho", "corr gam_o/rho_o"))
_ok = np.isfinite(rho_ratio_var)
for level in LEVELS:
    a = RPC[level]["as-is"]
    m = _ok & np.isfinite(a)
    print("%-14s %12.3f %12.3f"
          % (level, np.corrcoef(a[m], rho_ratio_var[m])[0, 1],
             np.corrcoef(GAM_O[level]["as-is"][m], rho_o_var.to_numpy()[m])[0, 1]))